# 🚦 Day 03a: Flow & Congestion Control — TCP Traffic Management

---

## 🎯 What You'll Master Today
- Flow control (sliding window)
- Congestion control (slow start, AIMD)
- Fast retransmit and fast recovery
- cwnd vs rwnd

---

### 🎭 The Analogy

**Flow control** = don't pour water faster than the glass can hold (receiver buffer).  
**Congestion control** = don't pour more cars on a highway than it can handle (network capacity).

```
╔══════════════════════════════════════════════════════════════════════╗
║          FLOW CONTROL vs CONGESTION CONTROL                         ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  FLOW CONTROL:                                                       ║
║    Sender  ────────────→  Receiver                                  ║
║    "How fast can YOU handle data?"                                   ║
║    Mechanism: rwnd (receiver window) advertised in TCP header        ║
║                                                                      ║
║  CONGESTION CONTROL:                                                 ║
║    Sender  ────[network]────→  Receiver                             ║
║    "How much can the NETWORK handle?"                               ║
║    Mechanism: cwnd (congestion window) maintained by sender          ║
║                                                                      ║
║  Effective window = min(cwnd, rwnd)                                 ║
║  Send up to this many bytes before waiting for ACK                  ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Sliding Window Simulator
# ============================================

class SlidingWindow:
    def __init__(self, window_size, total_packets):
        self.window = window_size
        self.total = total_packets
        self.base = 0
        self.next_seq = 0
    
    def simulate(self):
        print(f"  ═══ Sliding Window (size={self.window}, packets={self.total}) ═══\n")
        round_num = 1
        
        while self.base < self.total:
            # Send packets within window
            sent = []
            while self.next_seq < self.base + self.window and self.next_seq < self.total:
                sent.append(self.next_seq)
                self.next_seq += 1
            
            # Visualize window
            window_viz = ""
            for i in range(self.total):
                if i < self.base:
                    window_viz += "✅"  # ACKed
                elif i in sent:
                    window_viz += "📤"  # In-flight
                else:
                    window_viz += "⬜"  # Not yet sent
            
            print(f"    Round {round_num}: {window_viz}")
            print(f"             base={self.base} sent={sent} window=[{self.base}:{self.base+self.window})")
            
            # ACK all sent packets (simplified)
            self.base = self.next_seq
            round_num += 1
            print()
        
        print(f"    All {self.total} packets delivered in {round_num-1} rounds!")
        print(f"    Without sliding window: {self.total} rounds (stop-and-wait)")

sw = SlidingWindow(window_size=4, total_packets=12)
sw.simulate()

In [ ]:
# ============================================
# 2. TCP Congestion Control — Slow Start + AIMD
# ============================================

def tcp_congestion_control(max_rounds=20, ssthresh_init=16, loss_at=None):
    """Simulate TCP Reno congestion control."""
    cwnd = 1  # Start with 1 MSS
    ssthresh = ssthresh_init
    phase = "Slow Start"
    history = []
    
    print(f"  ═══ TCP Congestion Control (ssthresh={ssthresh_init}) ═══\n")
    print(f"  {'Round':>5} {'cwnd':>6} {'ssthresh':>10} {'Phase':>20}")
    print(f"  {'─'*5} {'─'*6} {'─'*10} {'─'*20}")
    
    for r in range(1, max_rounds + 1):
        # Check for loss
        if loss_at and r == loss_at:
            print(f"  {r:>5} {cwnd:>6} {ssthresh:>10} {'❌ PACKET LOSS!':>20}")
            ssthresh = max(cwnd // 2, 2)
            cwnd = 1  # Slow start (TCP Tahoe) or cwnd/2 (TCP Reno)
            phase = "Slow Start (after loss)"
            history.append((r, cwnd, phase))
            continue
        
        # Determine phase
        if cwnd < ssthresh:
            phase = "Slow Start (×2)"
            history.append((r, cwnd, phase))
            print(f"  {r:>5} {cwnd:>6} {ssthresh:>10} {phase:>20}")
            cwnd *= 2  # Exponential growth
        else:
            phase = "Congestion Avoidance (+1)"
            history.append((r, cwnd, phase))
            print(f"  {r:>5} {cwnd:>6} {ssthresh:>10} {phase:>27}")
            cwnd += 1  # Linear growth (AIMD)
    
    return history

history = tcp_congestion_control(max_rounds=15, ssthresh_init=16, loss_at=10)

print("\n  Phases:")
print("    1. Slow Start: cwnd doubles each RTT (exponential)")
print("    2. Congestion Avoidance: cwnd += 1 each RTT (linear, AIMD)")
print("    3. On loss: ssthresh = cwnd/2, cwnd = 1 (TCP Tahoe)")
print("                or cwnd = cwnd/2 (TCP Reno — fast recovery)")

In [ ]:
# ============================================
# 3. ASCII Art — cwnd Over Time
# ============================================

def visualize_cwnd():
    # Simulate simple cwnd progression with loss
    cwnds = [1, 2, 4, 8, 16, 17, 18, 19, 20, 21, 22]  # slow start → CA
    # Loss at cwnd=22: ssthresh=11, cwnd=1
    cwnds += [1, 2, 4, 8, 11, 12, 13, 14, 15]  # restart
    
    max_cwnd = max(cwnds)
    print("  ═══ cwnd Over Time (TCP Tahoe) ═══\n")
    print(f"  cwnd")
    
    for level in range(max_cwnd, 0, -1):
        row = f"  {level:>3} │"
        for c in cwnds:
            if c >= level:
                row += "█"
            else:
                row += " "
        print(row)
    
    print(f"  {'':>3} └{'─' * len(cwnds)}→ time (RTTs)")
    print(f"  {'':>3}  {'':>4}↑SS{'':>4}↑CA{'':>5}↑Loss{'':>1}↑SS{'':>2}↑CA")

visualize_cwnd()

print("\n  SS = Slow Start (exponential), CA = Congestion Avoidance (linear)")
print("  AIMD = Additive Increase, Multiplicative Decrease")

In [ ]:
# ============================================
# 4. TCP Variants Comparison
# ============================================

print("""
  ╔══════════════════════════════════════════════════════════════════╗
  ║                  TCP CONGESTION VARIANTS                        ║
  ╠══════════════════════════════════════════════════════════════════╣
  ║  Variant  │ On 3 dup ACKs     │ On timeout        │ Notes      ║
  ║  ─────────┼───────────────────┼───────────────────┼────────── ║
  ║  Tahoe    │ ssthresh=cwnd/2   │ ssthresh=cwnd/2   │ Original  ║
  ║           │ cwnd=1 (restart)  │ cwnd=1 (restart)  │           ║
  ║           │                   │                    │           ║
  ║  Reno     │ ssthresh=cwnd/2   │ ssthresh=cwnd/2   │ + Fast   ║
  ║           │ cwnd=cwnd/2       │ cwnd=1 (restart)  │ Recovery  ║
  ║           │ (fast recovery!)  │                    │           ║
  ║           │                   │                    │           ║
  ║  CUBIC    │ cwnd grows as     │ Same as Reno      │ Linux    ║
  ║           │ cubic function    │                    │ default  ║
  ║           │ (faster recovery) │                    │           ║
  ║           │                   │                    │           ║
  ║  BBR      │ Based on measured │ Based on measured  │ Google   ║
  ║  (Google) │ bandwidth + RTT   │ bandwidth + RTT    │ model-   ║
  ║           │ (doesn't use loss)│ (doesn't use loss) │ based    ║
  ╚══════════════════════════════════════════════════════════════════╝

  Key insight:
  - Tahoe: always restarts cwnd to 1 (too aggressive)
  - Reno: on 3 dup ACKs, cwnd halved (fast recovery)
  - CUBIC: cubic function for faster bandwidth recovery
  - BBR: measures actual bandwidth, doesn't rely on loss signals
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Flow control vs congestion control? | Flow: receiver capacity (rwnd). Congestion: network capacity (cwnd) |
| 2 | What is slow start? | cwnd starts at 1, doubles each RTT (exponential) until ssthresh |
| 3 | What is AIMD? | Additive Increase (cwnd+=1 per RTT), Multiplicative Decrease (cwnd/=2 on loss) |
| 4 | Fast retransmit vs timeout? | 3 dup ACKs → retransmit immediately (don't wait for timeout). Much faster recovery |
| 5 | TCP Tahoe vs Reno? | Tahoe: always cwnd=1 on loss. Reno: cwnd=cwnd/2 on 3 dup ACKs (fast recovery) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Effective window = min(cwnd, rwnd)                     │
## │  • Slow start: exponential growth (cwnd ×2 per RTT)      │
## │  • Congestion avoidance: linear growth (cwnd +1 per RTT) │
## │  • AIMD: additive increase, multiplicative decrease      │
## │  • Fast retransmit: 3 dup ACKs → don't wait for timeout │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **IP Addressing** — IPv4 classes, CIDR, subnetting